# Overlap-save convolution and FDAF verification

驗證完整 real/complex convolution、非整 block 長度的 frozen FDAF，以及頻域 adaptive identification。

來源：`OverLapSaveFDAF-doctest.ipynb`。所有資料與結果留在記憶體，圖表使用共用深色風格。

執行順序：設定 → 參數與資料 → 模型與訓練 → 圖表與數值摘要。

## 1. Setup

In [ ]:
from pathlib import Path
import sys
from functools import partial

import numpy as np

project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "adsp").is_dir())
if str(project_root) not in sys.path:
  sys.path.insert(0, str(project_root))

from adsp import data, training, statistics as stats
from adsp import plotting as plot
from adsp import filters as af
from adsp import equalization as ae
from adsp.datasets import SampleBatch

plot.use_dark()

## 2. Full convolution and frozen block filter

In [ ]:
from adsp.convolution import olsFFTConv
from scipy.signal import lfilter

seed, n_samples, n_taps = 55, 513, 16
rng = np.random.default_rng(seed)
x = data.gaussian(n_samples, rng=rng, complex_data=True)
h = data.gaussian(n_taps, rng=rng, complex_data=True, variance=0.1)
full = olsFFTConv(x, h, N=64)
assert np.allclose(full, np.convolve(x, h), atol=1e-10)
batch = SampleBatch(x, lfilter(h, [1.0], x))
frozen = training.run(af.OverLapSaveFDAF(filter_order=n_taps - 1, mu=0.1, init_coef=h), batch, adapt=False,
                      name="Frozen FDAF")
assert len(frozen.y) == n_samples
assert np.allclose(frozen.y, batch.d, atol=1e-10)
print("Full convolution and final partial block passed.")

## 3. Adaptive system identification

In [ ]:
h = np.r_[0.5, -0.3, 0.15, np.zeros(29)]
make_data = partial(data.system_identification, 4097, h=h, noise_var=1e-4)
models = {"FDAF": partial(af.OverLapSaveFDAF, filter_order=31, mu=0.5, alpha=0.9)}
results = training.compare(models, make_data, n_trials=5, seed=seed)
result = results["FDAF"].representative
assert len(result.y) == 4097
plot.learning(results, window=65, noise_floor=1e-4)
plot.show()
plot.stem({"Plant": h, "Learned physical FDAF taps": result.w_final.real}, title="FDAF coefficient convention")
plot.show()
training.summary(results, tail=1000)